In [1]:
!pip install -q torchaudio

import os
from os import path
import pandas as pd
import zipfile
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
import timm
import torchaudio.transforms as T
from sklearn.metrics import roc_curve, accuracy_score
from scipy.optimize import brentq
from scipy.interpolate import interp1d
import numpy as np

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Compute Device: {device}")
if device.type == 'cuda':
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

Compute Device: cuda
GPU Model: Tesla T4


In [2]:
class Config:
    # Audio Parameters
    SR = 16000
    MAX_DURATION = 4.0 # seconds
    MAX_SAMPLES = int(SR * MAX_DURATION)

    # Spectrogram / Feature Extraction
    N_FFT = 1024
    HOP_LENGTH = 256
    N_MELS = 80
    N_LFCC = 60

    # Derived Dimensions for Tensors
    # Time frames = MAX_SAMPLES // HOP_LENGTH + 1 = 64000 // 256 + 1 = 251
    TIME_FRAMES = int(MAX_SAMPLES / HOP_LENGTH) + 1

    # Model Architecture
    IN_CHANNELS = 2 # Channel 0: Mel, Channel 1: LFCC
    EMBED_DIM = 512
    FUSED_DIM = 1024

    # Training
    BATCH_SIZE = 16
    LEARNING_RATE = 1e-4

print(f"Config Loaded. Expected Temporal Frames: {Config.TIME_FRAMES}")

Config Loaded. Expected Temporal Frames: 251


In [3]:
import torchaudio.transforms as T
import torchaudio.functional as AF
import random
import torch
import torch.nn as nn

# Notice: 'import torchaudio.io' is removed from up here!

class TelephonySimulator(nn.Module):
    def __init__(self, sample_rate=16000, p=0.8):
        super().__init__()
        self.sample_rate = sample_rate
        self.p = p 

    def apply_phase_perturbation(self, waveform):
        spec = torch.fft.rfft(waveform)
        mag, phase = spec.abs(), spec.angle()
        phase_jitter = (torch.rand_like(phase) - 0.5) * 0.1
        new_spec = torch.polar(mag, phase + phase_jitter)
        return torch.fft.irfft(new_spec, n=waveform.shape[-1])

    def apply_isd_noise(self, waveform, P=20.0, g_sd=2.0):
        prob = torch.empty(1).uniform_(0, P).item() / 100.0
        mask = torch.rand_like(waveform) < prob
        
        f_r = (2.0 * torch.rand_like(waveform) - 1.0) * (2.0 * torch.rand_like(waveform) - 1.0)
        r = g_sd * waveform * f_r
        
        y = torch.where(mask, waveform + r, waveform)
        
        max_val = y.abs().max()
        return y / max_val if max_val > 1.0 else y

    def apply_ssi_noise(self, waveform, snr_min=5, snr_max=20):
        noise = torch.randn_like(waveform)
        snr_target = torch.empty(1).uniform_(snr_min, snr_max).item()
        
        sig_power = torch.norm(waveform, p=2)
        noise_power = torch.norm(noise, p=2)
        
        scale = (sig_power / (noise_power + 1e-8)) / (10.0 ** (snr_target / 20.0))
        noisy_waveform = waveform + noise * scale
        
        max_val = noisy_waveform.abs().max()
        return noisy_waveform / max_val if max_val > 1.0 else noisy_waveform

    def forward(self, waveform):
        if random.random() > self.p:
            return waveform

        if random.random() < 0.3:
            target_sr = random.choice([8000, 12000, 16000])
            resampler_down = T.Resample(self.sample_rate, target_sr).to(waveform.device)
            resampler_up = T.Resample(target_sr, self.sample_rate).to(waveform.device)
            waveform = resampler_down(waveform)
            waveform = resampler_up(waveform)

        waveform = waveform * random.uniform(0.8, 1.2)
        waveform = self.apply_phase_perturbation(waveform)
            
        waveform = AF.lowpass_biquad(waveform, self.sample_rate, cutoff_freq=random.uniform(3000, 3400))
        waveform = AF.highpass_biquad(waveform, self.sample_rate, cutoff_freq=random.uniform(200, 500))
        
        if random.random() < 0.5:
            waveform = self.apply_isd_noise(waveform, P=20.0)
        else:
            waveform = self.apply_ssi_noise(waveform, snr_min=5, snr_max=20)
        
        waveform = torch.clamp(waveform, -1.0, 1.0)
        waveform = AF.mu_law_encoding(waveform, quantization_channels=256).float()
        waveform = AF.mu_law_decoding(waveform, quantization_channels=256)

        return torch.clamp(waveform, min=-1.0, max=1.0)

class CodecSimulator(nn.Module):
    def __init__(self, sample_rate=16000, p=0.5):
        super().__init__()
        self.sample_rate = sample_rate
        self.p = p
        
        self.codecs = [
            ("mp3", [16000, 24000, 32000, 64000, 128000]), 
            ("vorbis", [16000, 24000, 32000, 64000, 96000]) 
        ]

    def forward(self, waveform):
        if random.random() > self.p:
            return waveform
            
        format, bitrates = random.choice(self.codecs)
        bitrate = random.choice(bitrates)
        
        try:
            # Attempt to import locally so it doesn't crash the whole cell if missing
            import torchaudio.io
            effector = torchaudio.io.AudioEffector(format=format, encoder=format, encoder_bit_rate=bitrate)
            compressed_waveform = effector.apply(waveform.T, self.sample_rate)
            return compressed_waveform.T
            
        except (ImportError, ModuleNotFoundError, RuntimeError):
            # KAGGLE FALLBACK: Pure PyTorch approximation of heavy compression
            
            # 1. Harsh lowpass filter (simulates MP3 high-freq cut-off)
            cutoff = random.uniform(3500, 5000) if bitrate <= 32000 else random.uniform(5000, 7000)
            waveform = AF.lowpass_biquad(waveform, self.sample_rate, cutoff_freq=cutoff)
            
            # 2. Add slight quantization noise (simulates bitrate reduction artifacts)
            noise_level = 0.005 if bitrate <= 32000 else 0.001
            noise = torch.randn_like(waveform) * noise_level
            waveform = waveform + noise
            
            return torch.clamp(waveform, -1.0, 1.0)

In [4]:
class AudioPreprocessor:
    def __init__(self, config):
        self.config = config
        self.mel_transform = T.MelSpectrogram(sample_rate=config.SR, n_fft=config.N_FFT, 
                                              hop_length=config.HOP_LENGTH, n_mels=config.N_MELS)
        self.amplitude_to_db = T.AmplitudeToDB()
        self.lfcc_transform = T.LFCC(sample_rate=config.SR, n_filter=config.N_MELS, n_lfcc=config.N_LFCC,
                                     speckwargs={"n_fft": config.N_FFT, "hop_length": config.HOP_LENGTH})
        
        # --- INITIALIZE BOTH SIMULATORS ---
        self.telephony_sim = TelephonySimulator(sample_rate=config.SR)
        self.codec_sim = CodecSimulator(sample_rate=config.SR, p=1.0) # Set p=1.0 here because we handle probability in the Dataset

    def normalize(self, tensor):
        return (tensor - tensor.mean()) / (tensor.std() + 1e-6)

In [5]:
import torch
import torch.nn as nn
import random

class GuidedSpecAugmentPolicy(nn.Module):
    def __init__(self, freq_mask_param=30, time_mask_param=60, num_masks=2, bias_low_freq=True):
        super().__init__()
        self.freq_mask_param = freq_mask_param
        self.time_mask_param = time_mask_param
        self.num_masks = num_masks
        self.bias_low_freq = bias_low_freq
        
        # Beta(1, 3) heavily favors values closer to 0 (lower frequency indices)
        self.beta_dist = torch.distributions.Beta(1.0, 3.0)

    def forward(self, x):
        # x shape: [Batch, Channels, Freq, Time]
        if not self.training:
            return x

        B, C, F, T_len = x.shape
        # Clone to avoid in-place modification issues during backprop
        x_aug = x.clone()

        for _ in range(self.num_masks):
            for b in range(B):
                # 1. Guided Frequency Masking
                f_mask_len = random.randint(1, self.freq_mask_param)
                
                if self.bias_low_freq:
                    # Sample from Beta distribution and scale to available frequency bins
                    f0 = int(self.beta_dist.sample().item() * (F - f_mask_len))
                else:
                    # Fallback to standard uniform random
                    f0 = random.randint(0, F - f_mask_len)
                    
                # Apply mask (0.0) across all channels and time steps for this frequency band
                x_aug[b, :, f0:f0 + f_mask_len, :] = 0.0

                # 2. Standard Time Masking (Uniform Random)
                t_mask_len = random.randint(1, self.time_mask_param)
                if T_len > t_mask_len:
                    t0 = random.randint(0, T_len - t_mask_len)
                    # Apply mask across all channels and frequencies for this time band
                    x_aug[b, :, :, t0:t0 + t_mask_len] = 0.0

        return x_aug

In [6]:
class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        # This is where the "adversarial" magic happens
        return grad_output.neg() * ctx.alpha, None

def grad_reverse(x, alpha=1.0):
    return GradientReversal.apply(x, alpha)

In [7]:
import math

class LocalTemporalBranch(nn.Module):
    def __init__(self, in_channels=2, embed_dim=512):
        super().__init__()
        self.resnet = models.resnet18(weights="DEFAULT")

        # UPGRADE 1: Smaller kernel, stride=1 to preserve high-frequency resolution
        self.resnet.conv1 = nn.Conv2d(
            in_channels, 64, kernel_size=3, stride=1, padding=1, bias=False
        )

        # FIX: Properly initialize the new 3x3 layer from scratch
        nn.init.kaiming_normal_(self.resnet.conv1.weight, mode='fan_out', nonlinearity='relu')
        
        # UPGRADE 2: Delete the MaxPool layer so we don't blur the AR artifacts
        self.resnet.maxpool = nn.Identity()
            
        self.feature_extractor = nn.Sequential(*list(self.resnet.children())[:-2])

        self.bilstm = nn.LSTM(
            input_size=512,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        
        # CHANGED: Use Max Pooling instead of Mean to preserve sharp AR artifacts
        x, _ = torch.max(x, dim=2) 
        
        x = x.permute(0, 2, 1)
        lstm_out, _ = self.bilstm(x)
        embedding = lstm_out[:, -1, :] 
        return embedding

class AttentiveStatisticsPooling(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Conv1d(in_dim, 128, kernel_size=1),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Conv1d(128, in_dim, kernel_size=1),
            nn.Softmax(dim=2) # Softmax across time
        )

    def forward(self, x):
        # x shape: [Batch, Time, Features] -> change to [Batch, Features, Time] for Conv1d
        x = x.transpose(1, 2)
        
        # Calculate attention weights for each frame
        alpha = self.attention(x)
        
        # Weighted mean
        means = torch.sum(alpha * x, dim=2)
        
        # Weighted standard deviation
        residuals = x - means.unsqueeze(2)
        variances = torch.sum(alpha * (residuals ** 2), dim=2)
        stds = torch.sqrt(variances.clamp(min=1e-9))
        
        # Concatenate mean and std [Batch, Features * 2]
        return torch.cat([means, stds], dim=1)

from transformers import WavLMModel
import torch
import torch.nn as nn
import torch.nn.functional as F

class GlobalContextBranch(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        self.wavlm = WavLMModel.from_pretrained("microsoft/wavlm-base-plus", output_hidden_states=True)
        self.layer_weights = nn.Parameter(torch.ones(13)) 
        
        # UPGRADE: Add ASP
        self.asp = AttentiveStatisticsPooling(in_dim=768)
        
        # ASP outputs Mean+Std (768 * 2 = 1536)
        self.proj = nn.Linear(1536, embed_dim) 

    def forward(self, waveform):
        if waveform.dim() == 3: waveform = waveform.squeeze(1)
        waveform = (waveform - waveform.mean(dim=-1, keepdim=True)) / (waveform.std(dim=-1, keepdim=True) + 1e-7)
            
        outputs = self.wavlm(waveform)
        all_layers = torch.stack(outputs.hidden_states) 
        weights = F.softmax(self.layer_weights, dim=0)

        if self.training:
            drop_mask = (torch.rand_like(weights) > 0.1).float()
            weights = (weights * drop_mask) / ((weights * drop_mask).sum() + 1e-8)
        
        fused_layers = (all_layers * weights.view(-1, 1, 1, 1)).sum(dim=0)
        
        # Pass the fused sequence [Batch, Time, 768] through ASP
        pooled_output = self.asp(fused_layers)
        
        return self.proj(pooled_output)

class AttentionFusion(nn.Module):
    def __init__(self, embed_dim=512, modality_dropout=0.1): 
        super().__init__()
        self.modality_dropout = modality_dropout
        self.attention = nn.Sequential(
            nn.Linear(embed_dim * 2, 128),
            nn.Tanh(),
            nn.Linear(128, 2), 
            nn.Softmax(dim=1)
        )

    def forward(self, embed1, embed2):
        # UPGRADE: Modality Dropout during training
        if self.training:
            if random.random() < self.modality_dropout:
                embed1 = torch.zeros_like(embed1)
            if random.random() < self.modality_dropout:
                embed2 = torch.zeros_like(embed2)

        stacked = torch.cat([embed1, embed2], dim=-1) 
        weights = self.attention(stacked) 
        
        weighted_embed1 = embed1 * weights[:, 0].unsqueeze(1)
        weighted_embed2 = embed2 * weights[:, 1].unsqueeze(1)
        
        return torch.cat([weighted_embed1, weighted_embed2], dim=-1)

class HybridDeepfakeDetector(nn.Module):
    def __init__(self, config, num_channels=2, k_subcenters=3):
        super().__init__()
        self.config = config
        self.branch1 = LocalTemporalBranch(in_channels=config.IN_CHANNELS, embed_dim=config.EMBED_DIM)
        self.branch2 = GlobalContextBranch(embed_dim=config.EMBED_DIM)
        self.fusion = AttentionFusion(embed_dim=config.EMBED_DIM)
        self.spec_dropout = GuidedSpecAugmentPolicy(freq_mask_param=20, time_mask_param=40, num_masks=2, bias_low_freq=True)
        self.latent_mask = nn.Dropout(p=0.2) 

        self.classifier = nn.Sequential(
            nn.Linear(config.FUSED_DIM, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.5)
        )

        self.channel_classifier = nn.Sequential(
            nn.Linear(config.FUSED_DIM, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_channels)
        )
        
        self.k_subcenters = k_subcenters
        # We now have 1 center for Bonafide (Real) and K centers for Spoof (Fake)
        self.margin_weight = nn.Parameter(torch.FloatTensor(1 + k_subcenters, 256))
        nn.init.xavier_uniform_(self.margin_weight)
        
        self.s, self.m = 30.0, 0.2
    def forward(self, waveform, mel, lfcc, labels=None, alpha=1.0):
        # We KEEP the Mel + LFCC stacked configuration!
        lfcc_aligned = F.interpolate(lfcc, size=(self.config.N_MELS, mel.shape[-1]), mode='bilinear', align_corners=False)
        x_stacked = torch.cat([mel, lfcc_aligned], dim=1)
        if self.training: x_stacked = self.spec_dropout(x_stacked)

        embed1 = self.branch1(x_stacked) 
        embed2 = self.branch2(waveform)     
        
        if self.training:
            embed1, embed2 = self.latent_mask(embed1), self.latent_mask(embed2)

        fused = self.fusion(embed1, embed2)
        
        rev_fused = grad_reverse(fused, alpha)
        channel_logits = self.channel_classifier(rev_fused)

        embeddings = self.classifier(fused)
        
        # Calculate cosine similarity to ALL centers (1 Real + K Fake)
        cos_theta_all = F.linear(F.normalize(embeddings), F.normalize(self.margin_weight))
        
        # Merge the K Spoof centers by taking the maximum similarity among them
        cos_bonafide = cos_theta_all[:, 0].unsqueeze(1)
        cos_spoof, _ = torch.max(cos_theta_all[:, 1:], dim=1, keepdim=True)
        cos_theta = torch.cat([cos_bonafide, cos_spoof], dim=1)
        
        if self.training and labels is not None:
            labels = labels.view(-1).long()
            
            # Standard ArcFace math applied to the merged logits
            sine = torch.sqrt(1.0 - torch.pow(cos_theta, 2)).clamp(0, 1)
            phi = cos_theta * math.cos(self.m) - sine * math.sin(self.m)
            
            one_hot = torch.zeros(cos_theta.size(), device=embeddings.device)
            one_hot.scatter_(1, labels.view(-1, 1), 1.0)
            
            logits_m = (one_hot * phi) + ((1.0 - one_hot) * cos_theta)
            return self.s * logits_m, channel_logits
            
        return cos_theta

In [8]:
def set_backbone_trainable(model, trainable=False):

    m = model.module if isinstance(model, nn.DataParallel) else model
    
    # 1. Toggle ResNet backbone (Branch 1)
    for param in m.branch1.feature_extractor.parameters():
        param.requires_grad = trainable
        
    # 2. Toggle WavLM backbone securely
    for name, param in m.branch2.wavlm.named_parameters():
        # NEVER thaw the CNN feature extractor to prevent catastrophic forgetting
        if "feature_extractor" in name:
            param.requires_grad = False
        else:
            param.requires_grad = trainable
            
    state = "UNFROZEN (Transformer Only)" if trainable else "FROZEN (Locked)"
    print(f"Model Backbones are now: {state}")

In [9]:
def verify_pipeline():
    preprocessor = AudioPreprocessor(Config)
    
    waveform = torch.randn(1, Config.MAX_SAMPLES).to(device)

    model = HybridDeepfakeDetector(Config).to(device)
    model.eval()

    mel = torch.randn(1, 1, Config.N_MELS, Config.TIME_FRAMES).to(device)
    lfcc = torch.randn(1, 1, Config.N_LFCC, Config.TIME_FRAMES).to(device)

    with torch.no_grad():
        output = model(waveform, mel, lfcc)

    print(f"Verification Successful!")
    print(f"Input Waveform Shape: {waveform.shape}")
    print(f"Input Mel Shape: {mel.shape}")
    print(f"Input LFCC Shape: {lfcc.shape}")
    print(f"Model Output Shape: {output.shape} (Raw Logits)")

try:
    verify_pipeline()
except Exception as e:
    print(f"Pipeline check failed: {e}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 160MB/s]


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Verification Successful!
Input Waveform Shape: torch.Size([1, 64000])
Input Mel Shape: torch.Size([1, 1, 80, 251])
Input LFCC Shape: torch.Size([1, 1, 60, 251])
Model Output Shape: torch.Size([1, 2]) (Raw Logits)


In [10]:
import os
import random
import pandas as pd
import torch
import torchaudio
import torchaudio.transforms as T
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# --- 1. Hardcoded Paths (2019 ONLY) ---
root_2019 = Path("/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA")
train_meta = root_2019 / "ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
train_audio = root_2019 / "ASVspoof2019_LA_train/flac"
dev_meta = root_2019 / "ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt"
dev_audio = root_2019 / "ASVspoof2019_LA_dev/flac"

# --- 2. FAST Unified Metadata Parser ---
def load_data_split_fast(meta_path, audio_dir):
    rows = []
    if not meta_path.exists():
        print(f"Error: {meta_path} not found")
        return pd.DataFrame()

    print(f"Scanning directory {audio_dir.name} (FAST)...")
    
    # Instantly map all available files
    available_files = {}
    if audio_dir.exists():
        for filename in os.listdir(audio_dir):
            if filename.endswith(".flac"):
                file_id = filename.split('.')[0]
                available_files[file_id] = str(audio_dir / filename)

    # Parse metadata and match instantly
    with open(meta_path, 'r') as f:
        lines = f.readlines()
        total = len(lines)
        print(f"Mapping {total} metadata lines...")

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5: 
            continue
        
        file_id = parts[1]
        label = parts[4] 
        
        # O(1) Instant Dictionary Lookup
        if file_id in available_files:
            rows.append({
                'path': available_files[file_id], 
                'label_id': 1 if label == 'spoof' else 0
            })

    df = pd.DataFrame(rows)
    if len(df) > 0:
        print(f"  Label distribution: {df['label_id'].value_counts().to_dict()}")
    return df

print("Loading ASVspoof 2019 Train...")
train_df = load_data_split_fast(train_meta, train_audio)

print("\nLoading ASVspoof 2019 Dev (Validation)...")
val_df_2019 = load_data_split_fast(dev_meta, dev_audio)

print(f"\nTrain samples (2019): {len(train_df)}")
print(f"Val samples (2019): {len(val_df_2019)}")

# --- 3. Dataset & Loaders ---
class AudioDataset(Dataset):
    def __init__(self, df, preprocessor, is_train=False):
        self.df = df.reset_index(drop=True)
        self.preprocessor = preprocessor
        self.is_train = is_train

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        channel_id = 0 # Default: Clean
        try:
            waveform, sr = torchaudio.load(row['path'])
            if sr != self.preprocessor.config.SR:
                waveform = T.Resample(sr, self.preprocessor.config.SR)(waveform)
            waveform = torch.mean(waveform, dim=0, keepdim=True) if waveform.shape[0] > 1 else waveform
            
            # Pad/Trim 
            if waveform.shape[1] > self.preprocessor.config.MAX_SAMPLES:
                offset = random.randint(0, waveform.shape[1] - self.preprocessor.config.MAX_SAMPLES) if self.is_train else 0
                waveform = waveform[:, offset:offset + self.preprocessor.config.MAX_SAMPLES]
            else:
                waveform = F.pad(waveform, (0, self.preprocessor.config.MAX_SAMPLES - waveform.shape[1]))

            # --- WEIGHTED DAT AUGMENTATION ---
            if self.is_train:
                rand_chance = random.random()
                
                # 15% Chance: Apply Telephony Simulation
                if rand_chance < 0.15:
                    waveform = self.preprocessor.telephony_sim(waveform)
                    channel_id = 1 
                    
                # 35% Chance: Apply Codec Compression (Targeting C2 weakness)
                elif rand_chance < 0.50:
                    waveform = self.preprocessor.codec_sim(waveform)
                    channel_id = 1 
                    
                # 50% Chance: Do nothing (Clean Audio)
                else:
                    channel_id = 0

            mel = self.preprocessor.normalize(self.preprocessor.amplitude_to_db(self.preprocessor.mel_transform(waveform)))
            lfcc = self.preprocessor.normalize(self.preprocessor.lfcc_transform(waveform))
        except Exception as e:
            # Fallback for corrupted files
            waveform = torch.zeros((1, self.preprocessor.config.MAX_SAMPLES))
            mel = torch.zeros((1, 80, 251))
            lfcc = torch.zeros((1, 60, 251))
            channel_id = 0

        return waveform, mel, lfcc, torch.tensor([row['label_id']], dtype=torch.float32), torch.tensor(channel_id, dtype=torch.long)

try:
    preprocessor = AudioPreprocessor(Config)
    
    # KAGGLE OPTIMIZATION: Maximize CPU cores and pin memory for faster GPU transfer
    workers = 4 # Kaggle typically has 4 cores
    
    train_dataset = AudioDataset(train_df, preprocessor, is_train=True)
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, 
                              shuffle=True, num_workers=workers, pin_memory=True)
    
    val_dataset_2019_clean = AudioDataset(val_df_2019, preprocessor, is_train=False)
    val_loader_2019_clean = DataLoader(val_dataset_2019_clean, batch_size=Config.BATCH_SIZE, 
                                       shuffle=False, num_workers=workers, pin_memory=True)
    
    print("\nAll DataLoaders ready for training!")
except NameError as e:
    print(f"\nSetup error: {e}")

Loading ASVspoof 2019 Train...
Scanning directory flac (FAST)...
Mapping 25380 metadata lines...
  Label distribution: {1: 22800, 0: 2580}

Loading ASVspoof 2019 Dev (Validation)...
Scanning directory flac (FAST)...
Mapping 24844 metadata lines...
  Label distribution: {1: 22296, 0: 2548}

Train samples (2019): 25380
Val samples (2019): 24844

All DataLoaders ready for training!


In [11]:
def evaluate_model(model, dataloader, criterion, device):
    """
    Evaluates the model on a given dataset and returns Loss, Accuracy, and EER.
    """
    model.eval()
    val_loss = 0.0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        # Unpack waveform as well
        for waveform, mel, lfcc, labels, _ in dataloader:
            waveform, mel, lfcc, labels = waveform.to(device, non_blocking=True), mel.to(device, non_blocking=True), lfcc.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            # Forward pass now returns raw cos_theta (-1 to 1)
            with torch.amp.autocast('cuda'):
                cos_logits = model(waveform, mel, lfcc)

                m = model.module if isinstance(model, nn.DataParallel) else model
                scaled_logits = m.s * cos_logits
                loss = criterion(scaled_logits, labels.view(-1).long())
                
            val_loss += loss.item()

            # FIX: Drop the Softmax entirely! Use the raw cosine similarity 
            # for the spoof class (index 1) as the anomaly score.
            scores = cos_logits[:, 1]

            all_labels.extend(labels.cpu().numpy().flatten())
            all_preds.extend(scores.cpu().numpy().flatten())

    # Aggregate metrics
    avg_val_loss = val_loss / len(dataloader)
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    # Compute EER and Accuracy
    val_eer, thresh = compute_eer(all_labels, all_preds)
    
    # FIX: Since scores are now -1 to 1, a static 0.5 threshold won't work.
    # We use the optimal threshold found during the EER calculation instead.
    binary_preds = (all_preds >= thresh).astype(int)
    val_acc = accuracy_score(all_labels, binary_preds)

    return avg_val_loss, val_acc, val_eer

In [12]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import roc_curve, accuracy_score
from scipy.optimize import brentq
from scipy.interpolate import interp1d
import numpy as np

# --- 1. EER Calculation Helper ---
def compute_eer(y_true, y_scores):
    """
    Computes the Equal Error Rate (EER).
    y_true: Ground truth binary labels (0 = bonafide, 1 = spoof)
    y_scores: Predicted probabilities or logits
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_scores, pos_label=1)

    # EER is where False Positive Rate (FPR) == False Negative Rate (FNR)
    # FNR = 1 - True Positive Rate (TPR)
    eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    thresh = interp1d(fpr, thresholds)(eer)
    return eer, thresh


# --- 2. Training Loop Setup ---
def train_model(model, train_loader, val_loader_2019_clean, config, device, num_epochs=30, patience=5, resume_checkpoint=None, eval_interval=2):
    freeze_epochs = 5
    thaw_lr = 1e-5     
    
    # 1. Keep the class weights to handle the 10:1 dataset imbalance
    class_weights = torch.tensor([2.0, 1.0], dtype=torch.float).to(device) 
    
    # 2. DELETE WeightedFocalLoss. Use pure CrossEntropyLoss for ArcFace to stabilize.
    criterion_df = nn.CrossEntropyLoss(weight=class_weights).to(device)
    criterion_ch = nn.CrossEntropyLoss().to(device)

    m = model.module if isinstance(model, nn.DataParallel) else model
    
    backbone_params = list(m.branch1.feature_extractor.parameters()) + list(m.branch2.wavlm.parameters())
    backbone_ids = set(id(p) for p in backbone_params)
    head_params = [p for p in model.parameters() if id(p) not in backbone_ids]
    
    optimizer = optim.AdamW([
        {'params': head_params, 'lr': config.LEARNING_RATE},
        {'params': backbone_params, 'lr': config.LEARNING_RATE}  
    ], weight_decay=1e-4)
    
    scheduler = None
    best_val_eer = float('inf')
    epochs_no_improve = 0 
    start_epoch = 0

    scaler = torch.amp.GradScaler('cuda')

    # --- CHECKPOINT RESUMING BLOCK ---
    if resume_checkpoint and os.path.exists(resume_checkpoint):
        print(f"Loading checkpoint from '{resume_checkpoint}'...")
        checkpoint = torch.load(resume_checkpoint, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        try:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            print("Optimizer state restored successfully.")
        except (ValueError, KeyError) as e:
            print(f"Warning: Could not restore optimizer state ({e}). Using fresh optimizer.")
        
        start_epoch = checkpoint['epoch'] + 1
        if 'best_eer' in checkpoint:
            best_val_eer = checkpoint['best_eer']
        
        if start_epoch >= freeze_epochs:
            finetune_epochs = num_epochs - freeze_epochs
            
            # PROPER PLACEMENT: Initialize OneCycleLR for resuming
            scheduler = optim.lr_scheduler.OneCycleLR(
                optimizer, 
                max_lr=[config.LEARNING_RATE, thaw_lr],
                epochs=finetune_epochs, 
                steps_per_epoch=len(train_loader),
                pct_start=0.2,
                div_factor=10.0
            )
            
            # Fast-forward the OneCycle scheduler to the correct batch step
            batches_completed = (start_epoch - freeze_epochs) * len(train_loader)
            for _ in range(batches_completed):
                scheduler.step()
                
        print(f"Successfully restored! Resuming from Epoch {start_epoch+1}")
    else:
        print("Starting Training Loop from scratch...")

    # --- SINGLE, UNIFIED EPOCH LOOP ---
    for epoch in range(start_epoch, num_epochs):
        
        # --- Freeze/Thaw Logic ---
        if epoch == 0 or (epoch == start_epoch and epoch < freeze_epochs):
            print("\n--- PHASE 1: Feature Extraction (Backbones Frozen) ---")
            set_backbone_trainable(model, trainable=False)
        elif epoch == freeze_epochs:
            print("\n--- PHASE 2: Fine-Tuning (Backbones Thawed) ---")
            set_backbone_trainable(model, trainable=True)
            optimizer.param_groups[0]['lr'] = config.LEARNING_RATE  # head
            optimizer.param_groups[1]['lr'] = thaw_lr               # backbone
            print(f"Head LR: {config.LEARNING_RATE}, Backbone LR: {thaw_lr}")
            
            finetune_epochs = num_epochs - freeze_epochs
            
            # PROPER PLACEMENT: Initialize OneCycleLR for a fresh Phase 2 start
            scheduler = optim.lr_scheduler.OneCycleLR(
                optimizer, 
                max_lr=[config.LEARNING_RATE, thaw_lr], 
                epochs=finetune_epochs, 
                steps_per_epoch=len(train_loader),
                pct_start=0.2,    
                div_factor=10.0   
            )

        # --- Training Phase ---
        model.train() 
        running_df_loss = 0.0
        running_ch_loss = 0.0

        for batch_idx, (waveform, mel, lfcc, labels, channel_labels) in enumerate(train_loader): 
            p = float(batch_idx + epoch * len(train_loader)) / (num_epochs * len(train_loader))
            alpha = 2. / (1. + np.exp(-10 * p)) - 1
            waveform, mel, lfcc = waveform.to(device, non_blocking=True), mel.to(device, non_blocking=True), lfcc.to(device, non_blocking=True)
            labels, channel_labels = labels.to(device, non_blocking=True), channel_labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            
            with torch.amp.autocast('cuda'):
                logits, channel_logits = model(waveform, mel, lfcc, labels=labels, alpha=alpha)
                df_loss = criterion_df(logits, labels.view(-1).long())
                ch_loss = criterion_ch(channel_logits, channel_labels)
                total_loss = df_loss + ch_loss

            # --- AMP BACKWARD PASS ---
            scaler.scale(total_loss).backward()
            
            # Unscale before clipping gradients
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            scaler.update()
            
            # PROPER PLACEMENT: Only step the scheduler here. Do NOT initialize it here.
            if scheduler is not None and epoch >= freeze_epochs:
                scheduler.step()
            
            running_df_loss += df_loss.item()
            running_ch_loss += ch_loss.item()

            if batch_idx % 50 == 0:
                print(f"Batch [{batch_idx}/{len(train_loader)}] DF Loss: {df_loss.item():.4f} | CH Loss: {ch_loss.item():.4f}")

        avg_train_loss = (running_df_loss + running_ch_loss) / len(train_loader)
        
        current_lr = optimizer.param_groups[0]['lr']
        print(f"\nEpoch [{epoch+1}/{num_epochs}] Total Train Loss: {avg_train_loss:.4f} | LR: {current_lr:.2e}")

        # --- Conditional Evaluation Phase ---
        is_eval_epoch = ((epoch + 1) % eval_interval == 0) or ((epoch + 1) == num_epochs)

        if is_eval_epoch:
            print(f"--> Running Evaluation on Clean 2019 Val Set ({len(val_loader_2019_clean.dataset)} samples)...")
            
            avg_val_loss, val_acc, val_eer = evaluate_model(model, val_loader_2019_clean, criterion_df, device)
            
            print(f"Clean 2019 Val Loss: {avg_val_loss:.4f} | Val Accuracy: {val_acc*100:.2f}% | Val EER: {val_eer*100:.4f}%")

            if val_eer < best_val_eer:
                best_val_eer = val_eer
                epochs_no_improve = 0 
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'best_eer': best_val_eer,
                }, '/kaggle/working/best_hybrid_deepfake_model.pth')
                print(f"New best model saved with Clean 2019 EER: {best_val_eer*100:.4f}%")
            else:
                epochs_no_improve += 1
                print(f"Early Stopping Counter: {epochs_no_improve}/{patience}")
                if epochs_no_improve >= patience:
                    print(f"Early stopping triggered!")
                    break
        else:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
            }, '/kaggle/working/latest_hybrid_deepfake_model.pth')
            print("--> Skipped validation this epoch to save time. 'latest' checkpoint saved.")

        print("-" * 50)

    print("Training Complete!")
    return model

In [13]:
# Path to your saved model on Kaggle
saved_model_path = '/kaggle/working/best_hybrid_deepfake_model.pth'

model = HybridDeepfakeDetector(Config)

if torch.cuda.device_count() > 1:
    print(f"Distributed training to {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

model = model.to(device)

trained_model = train_model(
    model=model, 
    train_loader=train_loader, 
    val_loader_2019_clean=val_loader_2019_clean, # Updated parameter
    config=Config, 
    device=device,
    num_epochs=30,       
    patience=5,
    resume_checkpoint=saved_model_path,
    eval_interval=2      
)

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

Distributed training to 2 GPUs!
Starting Training Loop from scratch...

--- PHASE 1: Feature Extraction (Backbones Frozen) ---
Model Backbones are now: FROZEN (Locked)
Batch [0/1587] DF Loss: 5.3933 | CH Loss: 0.7043
Batch [50/1587] DF Loss: 4.5894 | CH Loss: 0.6540
Batch [100/1587] DF Loss: 2.0214 | CH Loss: 0.6038
Batch [150/1587] DF Loss: 1.4277 | CH Loss: 0.5266
Batch [200/1587] DF Loss: 3.3001 | CH Loss: 0.5554
Batch [250/1587] DF Loss: 1.4180 | CH Loss: 0.5005
Batch [300/1587] DF Loss: 2.9839 | CH Loss: 0.4177
Batch [350/1587] DF Loss: 0.6716 | CH Loss: 0.3847
Batch [400/1587] DF Loss: 1.6912 | CH Loss: 0.2566
Batch [450/1587] DF Loss: 1.1353 | CH Loss: 0.3286
Batch [500/1587] DF Loss: 0.3508 | CH Loss: 0.4697
Batch [550/1587] DF Loss: 0.9257 | CH Loss: 0.4071
Batch [600/1587] DF Loss: 1.6021 | CH Loss: 0.4232
Batch [650/1587] DF Loss: 0.0068 | CH Loss: 0.5612
Batch [700/1587] DF Loss: 0.0253 | CH Loss: 0.5080
Batch [750/1587] DF Loss: 0.6468 | CH Loss: 0.6017
Batch [800/1587] DF

# Eval

In [14]:
import os
import torch
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader

print("="*50)
print("PREPARING ASVSPOOF 2021 LA EVALUATION (FAST PARSE)")
print("="*50)

# --- 1. Define Paths for LA ---
test_meta_2021_la = Path("/kaggle/input/datasets/mohammedabdeldayem/avsspoof-2021/LA-keys-full/keys/LA/CM/trial_metadata.txt")

test_audio_2021_la_dirs = [
    Path("/kaggle/input/datasets/mohammedabdeldayem/avsspoof-2021/ASVspoof2021_LA_eval/ASVspoof2021_LA_eval/flac"),
    # Fallback in case the Kaggle uploader dumped files without the flac folder
    Path("/kaggle/input/datasets/mohammedabdeldayem/avsspoof-2021/ASVspoof2021_LA_eval/ASVspoof2021_LA_eval") 
]

# --- 2. FAST Directory Scanning ---
print("Scanning LA audio directories (this takes ~1 second)...")
file_to_path_la = {}
for directory in test_audio_2021_la_dirs:
    if directory.exists():
        for filename in os.listdir(directory):
            if filename.endswith(".flac"):
                file_id = filename.split('.')[0] 
                file_to_path_la[file_id] = directory / filename

# --- 3. FAST Metadata Parser ---
def load_la_data_split_fast(meta_path, path_dict):
    rows = []
    if not meta_path.exists():
        print(f"Error: {meta_path} not found")
        return pd.DataFrame()

    with open(meta_path, 'r') as f:
        lines = f.readlines()
        print(f"Mapping {len(lines)} LA metadata entries to files...")

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 6: 
            continue
        
        file_id = parts[1]
        label = parts[5] # 2021 labels are in the 6th column
        
        # O(1) Instant Dictionary Lookup instead of checking the hard drive
        if file_id in path_dict:
            rows.append({
                'path': str(path_dict[file_id]), 
                'label_id': 1 if label == 'spoof' else 0
            })

    df = pd.DataFrame(rows)
    if len(df) > 0:
        print(f"LA Eval Label distribution: {df['label_id'].value_counts().to_dict()}")
    return df

# --- 4. Load Metadata & Dataloader ---
val_df_2021_la = load_la_data_split_fast(test_meta_2021_la, file_to_path_la)

if len(val_df_2021_la) > 0:
    print(f"\nSuccessfully matched {len(val_df_2021_la)} LA audio files!")
    
    val_dataset_2021_la = AudioDataset(val_df_2021_la, preprocessor, is_train=False)
    val_loader_2021_la = DataLoader(
        val_dataset_2021_la, 
        batch_size=Config.BATCH_SIZE, 
        shuffle=False, 
        num_workers=4, 
        pin_memory=True
    )

    # --- 5. LA-Specific Evaluation Function ---
    def generate_la_cm_score_file(model, device, val_loader, val_df):
        print("\n" + "="*50)
        print("GENERATING CM SCORES FOR ASVspoof 2021 **LA** EVALUATION")
        print("="*50)
        
        print(f"\nEvaluating on ASVspoof 2021 LA Eval Set ({len(val_loader.dataset)} samples)...")
        
        # Ensure the passed model is in evaluation mode
        model.eval()
        all_preds = []

        with torch.no_grad():
            for batch_idx, (waveform, mel, lfcc, labels, _) in enumerate(val_loader): 
                waveform = waveform.to(device, non_blocking=True)
                mel = mel.to(device, non_blocking=True)
                lfcc = lfcc.to(device, non_blocking=True)
                
                with torch.amp.autocast('cuda'):
                    # Evaluate directly using the passed model
                    cos_logits = model(waveform, mel, lfcc)
                    scores = cos_logits[:, 1]
                
                all_preds.extend(scores.cpu().numpy().flatten())
                
                if batch_idx % 1000 == 0 and batch_idx > 0:
                    print(f"Evaluated {batch_idx * Config.BATCH_SIZE} files...")

        print("\nExporting LA predictions...")
        file_ids = val_df['path'].apply(lambda x: os.path.basename(x).replace('.flac', '')).tolist()
        
        output_path_original = '/kaggle/working/cm_scores_2021_LA_eval_original.txt'
        output_path_inverted = '/kaggle/working/cm_scores_2021_LA_eval_inverted.txt'
        
        with open(output_path_original, 'w') as f_orig, open(output_path_inverted, 'w') as f_inv:
            for file_id, score in zip(file_ids, all_preds):
                f_orig.write(f"{file_id} {score:.6f}\n")
                f_inv.write(f"{file_id} {-score:.6f}\n") 
                
        print(f"LA Original scores successfully saved to: {output_path_original}")
        print(f"LA Inverted scores successfully saved to: {output_path_inverted}")

    # --- 6. Run the evaluation ---
    # Passing the trained_model directly from memory
    try:
        generate_la_cm_score_file(trained_model, device, val_loader_2021_la, val_df_2021_la)
    except NameError:
        print("Error: 'trained_model' not found. Make sure you run your training cell first!")

else:
    print("Warning: Could not load LA metadata. Please check your Kaggle dataset paths.")

PREPARING ASVSPOOF 2021 LA EVALUATION (FAST PARSE)
Scanning LA audio directories (this takes ~1 second)...
Mapping 181566 LA metadata entries to files...
LA Eval Label distribution: {1: 163114, 0: 18452}

Successfully matched 181566 LA audio files!

GENERATING CM SCORES FOR ASVspoof 2021 **LA** EVALUATION

Evaluating on ASVspoof 2021 LA Eval Set (181566 samples)...
Evaluated 16000 files...
Evaluated 32000 files...
Evaluated 48000 files...
Evaluated 64000 files...
Evaluated 80000 files...
Evaluated 96000 files...
Evaluated 112000 files...
Evaluated 128000 files...
Evaluated 144000 files...
Evaluated 160000 files...
Evaluated 176000 files...

Exporting LA predictions...
LA Original scores successfully saved to: /kaggle/working/cm_scores_2021_LA_eval_original.txt
LA Inverted scores successfully saved to: /kaggle/working/cm_scores_2021_LA_eval_inverted.txt


In [15]:
import os
import torch
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader

print("="*50)
print("PREPARING ASVSPOOF 2021 DF EVALUATION (FAST PARSE & TTA)")
print("="*50)

# --- 1. Define Paths ---
test_meta_2021_df = Path("/kaggle/input/datasets/mohammedabdeldayem/avsspoof-2021/DF-keys-full/keys/DF/CM/trial_metadata.txt")

test_audio_2021_df_dirs = [
    Path("/kaggle/input/datasets/skenospeniel/asvspoof-21-df/ASVspoof2021_DF_eval_part00/ASVspoof2021_DF_eval/flac"),
    Path("/kaggle/input/datasets/skenospeniel/asvspoof-21-df/ASVspoof2021_DF_eval_part01/ASVspoof2021_DF_eval/flac"),
    Path("/kaggle/input/datasets/skenospeniel/asvspoof-21-df/ASVspoof2021_DF_eval_part02/ASVspoof2021_DF_eval/flac"),
    Path("/kaggle/input/datasets/skenospeniel/asvspoof-21-df/ASVspoof2021_DF_eval_part03/ASVspoof2021_DF_eval/flac") 
]

# --- 2. FAST Directory Scanning ---
print("Scanning audio directories (this takes ~2 seconds)...")
file_to_path = {}
for directory in test_audio_2021_df_dirs:
    if directory.exists():
        for filename in os.listdir(directory):
            if filename.endswith(".flac"):
                file_id = filename.split('.')[0] # e.g., 'DF_E_2000026'
                file_to_path[file_id] = directory / filename

# --- 3. FAST Metadata Parser ---
def load_df_data_split_fast(meta_path, path_dict):
    rows = []
    if not meta_path.exists():
        print(f"Error: {meta_path} not found")
        return pd.DataFrame()

    with open(meta_path, 'r') as f:
        lines = f.readlines()
        print(f"Mapping {len(lines)} metadata entries to files...")

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 6: 
            continue
        
        file_id = parts[1]
        label = parts[5] 
        
        # O(1) Instant Dictionary Lookup instead of .exists()
        if file_id in path_dict:
            rows.append({
                'path': str(path_dict[file_id]), 
                'label_id': 1 if label == 'spoof' else 0
            })

    df = pd.DataFrame(rows)
    if len(df) > 0:
        print(f"DF Eval Label distribution: {df['label_id'].value_counts().to_dict()}")
    return df

# --- 4. Load Metadata & Dataloader ---
val_df_2021_df = load_df_data_split_fast(test_meta_2021_df, file_to_path)

if len(val_df_2021_df) > 0:
    print(f"\nSuccessfully matched {len(val_df_2021_df)} audio files!")
    
    val_dataset_2021_df = AudioDataset(val_df_2021_df, preprocessor, is_train=False)
    val_loader_2021_df = DataLoader(
        val_dataset_2021_df, 
        batch_size=Config.BATCH_SIZE, 
        shuffle=False, 
        num_workers=4, 
        pin_memory=True
    )

    # --- 5. DF-Specific Evaluation Function with TTA ---
    def generate_df_cm_score_file(model, device, val_loader, val_df):
        print("\n" + "="*50)
        print("GENERATING CM SCORES FOR ASVspoof 2021 **DF** EVALUATION")
        print("="*50)
        
        print(f"\nEvaluating on ASVspoof 2021 DF Eval Set ({len(val_loader.dataset)} samples)...")
        print("Test-Time Augmentation (TTA) is ENABLED. This will process every file twice. Grab a coffee!")
        
        # Ensure the passed model is in evaluation mode
        model.eval()
        all_preds = []

        with torch.no_grad():
            for batch_idx, (waveform, mel, lfcc, labels, _) in enumerate(val_loader): 
                waveform = waveform.to(device, non_blocking=True)
                mel = mel.to(device, non_blocking=True)
                lfcc = lfcc.to(device, non_blocking=True)
                
                with torch.amp.autocast('cuda'):
                    # --- 1. Normal Evaluation ---
                    cos_logits_1 = model(waveform, mel, lfcc)
                    
                    # --- 2. TTA Evaluation (Slight temporal shift) ---
                    # Shift the waveform by 1000 samples to change the framing context
                    waveform_shifted = torch.roll(waveform, shifts=1000, dims=-1)
                    
                    # Compute complementary spectral representations on CPU to avoid torchaudio device crashes, then move back to GPU
                    mel_shifted = preprocessor.normalize(preprocessor.amplitude_to_db(preprocessor.mel_transform(waveform_shifted.cpu()))).to(device)
                    lfcc_shifted = preprocessor.normalize(preprocessor.lfcc_transform(waveform_shifted.cpu())).to(device)
                    
                    cos_logits_2 = model(waveform_shifted, mel_shifted, lfcc_shifted)
                    
                    # --- 3. Average the spoof scores of both passes ---
                    scores = (cos_logits_1[:, 1] + cos_logits_2[:, 1]) / 2.0
                
                all_preds.extend(scores.cpu().numpy().flatten())
                
                if batch_idx % 1000 == 0 and batch_idx > 0:
                    print(f"Evaluated {batch_idx * Config.BATCH_SIZE} files...")

        print("\nExporting predictions...")
        file_ids = val_df['path'].apply(lambda x: os.path.basename(x).replace('.flac', '')).tolist()
        
        output_path_original = '/kaggle/working/cm_scores_2021_DF_eval_original.txt'
        output_path_inverted = '/kaggle/working/cm_scores_2021_DF_eval_inverted.txt'
        
        with open(output_path_original, 'w') as f_orig, open(output_path_inverted, 'w') as f_inv:
            for file_id, score in zip(file_ids, all_preds):
                f_orig.write(f"{file_id} {score:.6f}\n")
                f_inv.write(f"{file_id} {-score:.6f}\n") 
                
        print(f"DF Original scores successfully saved to: {output_path_original}")
        print(f"DF Inverted scores successfully saved to: {output_path_inverted}")

    # --- 6. Run the evaluation ---
    try:
        generate_df_cm_score_file(trained_model, device, val_loader_2021_df, val_df_2021_df)
    except NameError:
        print("Error: 'trained_model' not found. Make sure you run your training cell first!")

else:
    print("Warning: Could not load DF metadata. Please check your Kaggle dataset paths.")

PREPARING ASVSPOOF 2021 DF EVALUATION (FAST PARSE & TTA)
Scanning audio directories (this takes ~2 seconds)...
Mapping 611829 metadata entries to files...
DF Eval Label distribution: {1: 589212, 0: 22617}

Successfully matched 611829 audio files!

GENERATING CM SCORES FOR ASVspoof 2021 **DF** EVALUATION

Evaluating on ASVspoof 2021 DF Eval Set (611829 samples)...
Test-Time Augmentation (TTA) is ENABLED. This will process every file twice. Grab a coffee!
Evaluated 16000 files...
Evaluated 32000 files...
Evaluated 48000 files...
Evaluated 64000 files...
Evaluated 80000 files...
Evaluated 96000 files...
Evaluated 112000 files...
Evaluated 128000 files...
Evaluated 144000 files...
Evaluated 160000 files...
Evaluated 176000 files...
Evaluated 192000 files...
Evaluated 208000 files...
Evaluated 224000 files...
Evaluated 240000 files...
Evaluated 256000 files...
Evaluated 272000 files...
Evaluated 288000 files...
Evaluated 304000 files...
Evaluated 320000 files...
Evaluated 336000 files...
Ev